In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Cargar modelo entrenado
model_path = "final/instance_segmentation_model_V1.h5"  # O usa "model_keras.keras" si es el correcto
model = load_model(model_path)

# Definir clases y colores (cambia según tu modelo)
CLASSES = ["Fondo", 'Taza', 'Onepiece', 'Tanque', 'Lavamanos', 'Pedestal']
COLORS = [(0, 0, 0), (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

# Cargar imagen de prueba
image_path = "final/data/35.png"  # Ruta de la imagen
image = cv2.imread(image_path)
input_size = (image.shape[1], image.shape[0])  
image_resized = cv2.resize(image, (576, 768))  
input_tensor = np.expand_dims(image_resized / 255.0, axis=0)  # Normalizar y añadir batch

# Realizar inferencia
predictions = model.predict(input_tensor) 
mask = np.argmax(predictions[0], axis=-1)  # Seleccionar la clase más probable por píxel

# Redimensionar la máscara al tamaño original
mask_resized = cv2.resize(mask.astype(np.uint8), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

# Crear imagen de resultado
output = image.copy()

# Dibujar contornos e instancias
for class_id in range(1, len(CLASSES)):  # Omitir fondo
    class_mask = (mask_resized == class_id).astype(np.uint8) * 255
    contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        cv2.drawContours(output, [contour], -1, COLORS[class_id], 2)
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output, (x, y), (x + w, y + h), COLORS[class_id], 2)
        cv2.putText(output, CLASSES[class_id], (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS[class_id], 2)

# Mostrar imagen con máscaras y bounding boxes
cv2.imshow("Segmentación de instancias", output)
cv2.waitKey(0)
cv2.destroyAllWindows()


1/1 [==============================] - 1s 1s/step
